# Abstención

**Fase:** E16

## Objetivo

Analizar cuándo el agente se abstiene, y separar la abstención correcta de la injustificada.

## Relación con la pregunta de investigación

> ¿Puede un agente con RAG explicar el diagnóstico de un vehículo a partir de datos OBD-II reales, rastreando cada explicación hasta su evidencia de origen, y reconocer cuándo la evidencia disponible no es suficiente para sostener una explicación en lugar de generar una hipótesis sin fundamento?

La abstención es una salida válida del sistema, no un fallo. La PI pregunta si el agente sabe cuándo callar.

## Hipótesis

_Por declarar antes de ejecutar la fase._

## Datos

_Dataset, versión y partición utilizados._

## Método

_Procedimiento, parámetros y semillas._

## Métricas

_Qué se mide y con qué instrumento._

## Criterio de éxito o refutación

_Fijado antes de mirar los resultados._


In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd

from mecanico_ia_exp.config import paths


## El experimento central

Mismos datos, mismo modelo, misma atribución, **misma confianza**. Lo único que cambia es que se retira la evidencia del índice.

In [2]:
import json
s = json.loads((paths.EXPERIMENTS / 'E15_suficiencia.json').read_text())
par = [c for c in s['cases'] if 'vacuum_leak' in c['case_id']]
pd.DataFrame([{'caso': c['case_id'],
               'variante': c.get('corpus_variant', 'FULL'),
               'confianza': c['verdict']['signals']['confidence'],
               'docs aplicables': c['verdict']['signals']['n_documents_applicable'],
               'cobertura': c['verdict']['signals']['claim_coverage'],
               'decisión': c['verdict']['decision']} for c in par])

,caso,variante,confianza,docs aplicables,cobertura,decisión
0,C05_vacuum_leak,FULL,0.9078,4,1.0,EXPLAIN
1,C06_vacuum_leak_evidencia_retirada,ABSENT,0.9078,0,0.0,ABSTAIN


### Lo que el agente respondería en cada caso

In [3]:
for c in par:
    v = c['verdict']
    print(f"--- {c['case_id']}")
    if v['decision'] == 'EXPLAIN':
        for cl in c['bundle']['claims']:
            print(f"  [{cl['support_status']}] {cl['text'][:88]}")
            print(f"      observación: {cl['observational_evidence']}")
            print(f"      documento  : {cl['documentary_evidence']}")
    else:
        print('  ABSTAIN — INSUFFICIENT_EVIDENCE')
        for m in v['missing_evidence']:
            print(f'      falta: {m}')
    print()

--- C05_vacuum_leak
  [SUPPORTED] La señal ltft_bank1 (max) se sitúa por encima de lo esperable en la ventana observada, c
      observación: ['OBS001']
      documento  : ['mazda3-bk-workshop-manual-local@9a90d2f6a951:001851', 'mazda3-bk-workshop-manual-local@9a90d2f6a951:001835']
  [SUPPORTED] La señal stft_bank1 (mean) se sitúa por encima de lo esperable en la ventana observada, 
      observación: ['OBS002']
      documento  : ['mazda3-bk-workshop-manual-local@9a90d2f6a951:001851', 'mazda3-bk-workshop-manual-local@9a90d2f6a951:001835']
  [SUPPORTED] La señal ltft_bank1 (mean) se sitúa por encima de lo esperable en la ventana observada, 
      observación: ['OBS003']
      documento  : ['mazda3-bk-workshop-manual-local@9a90d2f6a951:001851', 'mazda3-bk-workshop-manual-local@9a90d2f6a951:001835']

--- C06_vacuum_leak_evidencia_retirada
  ABSTAIN — INSUFFICIENT_EVIDENCE
      falta: ningún fragmento documental aplicable a este vehículo
      falta: cobertura 0.00 < 0.80



> La abstención **no es un fallo**. Es la respuesta correcta cuando la evidencia disponible no sostiene una explicación.

## Resultados

_Lo observado, sin interpretación._

## Interpretación

_Qué significa, y qué no significa._

## Limitaciones

_Qué queda fuera del alcance de lo mostrado._

## Conclusión experimental

_Respuesta al criterio fijado en la cabecera._

## Artefactos generados

_Ficheros producidos, con su ruta._
